# Jyo Data Prep — Transformer-Ready Feature Pipeline

**Goal:** Produce normalized, embedding-friendly feature parquets that a transformer-based demand forecasting model can consume directly.

**Differences vs the LightGBM pipeline in `notebooks/shubhro/`:**
- Continuous features are `StandardScaler`-normalized (transformers need this; LGBM does not)
- Categorical features get **vocab sizes** persisted so embedding tables can be sized correctly
- Train rows with missing `RoadType` are **dropped** (user's explicit rule), not imputed
- Temperature is imputed with **per-Weather median** (user's rule), not per-(day,hour) mean

Everything else (lags, rolling, OOF target encodings, boundary features, saturation) is carried forward from `feature_engineering_v2.ipynb` because those decisions are model-agnostic.

## Outputs
| File | Purpose |
|---|---|
| `processed/jyo_train_features.parquet` | Transformer training input |
| `processed/jyo_test_features.parquet`  | Transformer inference input |
| `processed/jyo_scalers.pkl`            | Fitted `StandardScaler`s for reproducibility |
| `processed/jyo_feature_meta.json`      | Embedding vocab sizes, continuous & binary column lists |

## 1. Setup

In [2]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE      = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR  = os.path.join(BASE, 'dataset')
PROCESSED = os.path.join(BASE, 'processed')
os.makedirs(PROCESSED, exist_ok=True)
print('BASE     :', BASE)
print('DATA_DIR :', DATA_DIR)
print('PROCESSED:', PROCESSED)

BASE     : /home/jyo/Desktop/Projects/yeeyyeeniggaass
DATA_DIR : /home/jyo/Desktop/Projects/yeeyyeeniggaass/dataset
PROCESSED: /home/jyo/Desktop/Projects/yeeyyeeniggaass/processed


## 2. Load raw data

In [3]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print('train_raw:', train_raw.shape)
print('test_raw :', test_raw.shape)
train_raw.head(3)

train_raw: (77299, 11)
test_raw : (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny


## 3. Missing value handling (user's rules)

**RoadType:** drop train rows where NaN, impute test rows with train-only per-geohash mode (with prefix-5 → prefix-4 → global fallback).

**Temperature:** fill NaN with median Temperature of the same `Weather` value (from train non-null rows). Fallback to global train median.

**Weather:** fill NaN by `(day, hour)` train-only mode, with cascade fallback.

We record `*_was_missing` flags on raw values **before** imputation.

In [4]:
# --- Record missing-indicator flags from RAW data ---
for d in (train_raw, test_raw):
    for col in ('RoadType', 'Temperature', 'Weather'):
        d[f'{col}_was_missing'] = d[col].isna().astype(np.int8)

# --- RoadType: drop NaN rows from TRAIN ---
n_before = len(train_raw)
train_raw = train_raw.dropna(subset=['RoadType']).reset_index(drop=True)
n_dropped = n_before - len(train_raw)
print(f'Dropped {n_dropped} train rows with NaN RoadType (kept {len(train_raw)})')

# --- RoadType: impute TEST using train-only per-geohash mode + prefix cascade ---
def det_mode(values):
    s = pd.Series(values).dropna().astype(str)
    if len(s) == 0:
        return None
    vc = s.value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

train_raw['_p4'] = train_raw['geohash'].str[:4]
train_raw['_p5'] = train_raw['geohash'].str[:5]
test_raw['_p4']  = test_raw['geohash'].str[:4]
test_raw['_p5']  = test_raw['geohash'].str[:5]

rt_gh   = train_raw.groupby('geohash')['RoadType'].agg(det_mode)
rt_p5   = train_raw.groupby('_p5')['RoadType'].agg(det_mode)
rt_p4   = train_raw.groupby('_p4')['RoadType'].agg(det_mode)
rt_glob = det_mode(train_raw['RoadType'])
print('Global train RoadType mode:', rt_glob)

def impute_roadtype(row):
    if pd.notna(row['RoadType']):
        return row['RoadType']
    v = rt_gh.get(row['geohash'])
    if v is None or (isinstance(v, float) and np.isnan(v)):
        v = rt_p5.get(row['_p5'])
    if v is None or (isinstance(v, float) and np.isnan(v)):
        v = rt_p4.get(row['_p4'])
    if v is None or (isinstance(v, float) and np.isnan(v)):
        v = rt_glob
    return v

test_raw['RoadType'] = test_raw.apply(impute_roadtype, axis=1)
assert test_raw['RoadType'].isna().sum() == 0
print('Test RoadType after imputation — NaN:', int(test_raw['RoadType'].isna().sum()))

Dropped 600 train rows with NaN RoadType (kept 76699)
Global train RoadType mode: Residential
Test RoadType after imputation — NaN: 0


In [5]:
# --- Weather: fill via train-only (day, hour) mode cascade ---
def _hour_from_ts(ts):
    return int(str(ts).split(':')[0])

for d in (train_raw, test_raw):
    d['_hour'] = d['timestamp'].astype(str).map(_hour_from_ts).astype(int)

tr_w = train_raw.dropna(subset=['Weather'])
w_dh = tr_w.groupby(['day', '_hour'])['Weather'].agg(det_mode)
w_h  = tr_w.groupby('_hour')['Weather'].agg(det_mode)
w_d  = tr_w.groupby('day')['Weather'].agg(det_mode)
w_g  = det_mode(train_raw['Weather'])

def fill_weather(df):
    f1 = pd.Series(df.set_index(['day','_hour']).index.map(w_dh), index=df.index)
    df['Weather'] = df['Weather'].fillna(f1)
    df['Weather'] = df['Weather'].fillna(df['_hour'].map(w_h))
    df['Weather'] = df['Weather'].fillna(df['day'].map(w_d))
    df['Weather'] = df['Weather'].fillna(w_g)
    return df

train_raw = fill_weather(train_raw)
test_raw  = fill_weather(test_raw)
print('Weather NaN — train:', int(train_raw['Weather'].isna().sum()),
      ' test:', int(test_raw['Weather'].isna().sum()))
print('Global train Weather mode:', w_g)

Weather NaN — train: 0  test: 0
Global train Weather mode: Sunny


In [6]:
# --- Temperature: fill with median per Weather (train-only), fallback global train median ---
tr_t = train_raw.dropna(subset=['Temperature'])
temp_per_weather = tr_t.groupby('Weather')['Temperature'].median()
temp_global      = float(tr_t['Temperature'].median())
print('Train Temperature median per Weather:')
print(temp_per_weather)
print(f'Global train Temperature median: {temp_global:.4f}')

def fill_temperature(df):
    f = df['Weather'].map(temp_per_weather)
    df['Temperature'] = df['Temperature'].fillna(f)
    df['Temperature'] = df['Temperature'].fillna(temp_global)
    return df

train_raw = fill_temperature(train_raw)
test_raw  = fill_temperature(test_raw)
print('Temperature NaN — train:', int(train_raw['Temperature'].isna().sum()),
      ' test:', int(test_raw['Temperature'].isna().sum()))

# Drop helper cols
for d in (train_raw, test_raw):
    d.drop(columns=['_p4','_p5','_hour'], inplace=True)

Train Temperature median per Weather:
Weather
Foggy    16.479014
Rainy    11.145024
Snowy     4.406222
Sunny    22.962484
Name: Temperature, dtype: float64
Global train Temperature median: 16.3830
Temperature NaN — train: 0  test: 0


## 4. Time features
Cyclic encodings are critical for a transformer that lacks an inductive bias for periodic signals.

In [7]:
def add_time_features(df):
    df = df.copy()
    parts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour']         = parts[0].astype(int).astype(np.int16)
    df['minute']       = parts[1].astype(int).astype(np.int16)
    df['qhour_of_day'] = (df['hour'] * 4 + df['minute'] // 15).astype(np.int16)
    df['t_abs']        = (df['day'] * 96 + df['qhour_of_day']).astype(np.int32)
    df['hour_sin']     = np.sin(2*np.pi*df['hour']/24).astype(np.float32)
    df['hour_cos']     = np.cos(2*np.pi*df['hour']/24).astype(np.float32)
    df['qhour_sin']    = np.sin(2*np.pi*df['qhour_of_day']/96).astype(np.float32)
    df['qhour_cos']    = np.cos(2*np.pi*df['qhour_of_day']/96).astype(np.float32)
    def tod_bucket(h):
        if   3  <= h <  9: return 'morning_ramp'
        elif 9  <= h < 14: return 'midday_plateau'
        elif 14 <= h < 19: return 'afternoon_drop'
        elif 19 <= h < 22: return 'evening_trough'
        else:              return 'late_night_rise'
    df['tod_bucket'] = df['hour'].apply(tod_bucket)
    return df

train = add_time_features(train_raw)
test  = add_time_features(test_raw)
print('train t_abs range:', train['t_abs'].min(), '->', train['t_abs'].max())
print('test  t_abs range:', test['t_abs'].min(),  '->', test['t_abs'].max())

train t_abs range: 4608 -> 4712
test  t_abs range: 4713 -> 4759


## 5. Geohash hierarchy features

In [8]:
for d in (train, test):
    d['gh_prefix4'] = d['geohash'].str[:4]
    d['gh_prefix5'] = d['geohash'].str[:5]

# --- Geohash -> lat/lon decoder (vendored) ---
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
_BITS   = [16, 8, 4, 2, 1]

def geohash_decode(gh):
    lat_lo, lat_hi = -90.0, 90.0
    lon_lo, lon_hi = -180.0, 180.0
    is_even = True
    for c in gh:
        cd = _BASE32.index(c)
        for mask in _BITS:
            if is_even:
                if cd & mask: lon_lo = (lon_lo + lon_hi) / 2
                else:         lon_hi = (lon_lo + lon_hi) / 2
            else:
                if cd & mask: lat_lo = (lat_lo + lat_hi) / 2
                else:         lat_hi = (lat_lo + lat_hi) / 2
            is_even = not is_even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.Index(pd.concat([train['geohash'], test['geohash']]).unique())
coord  = pd.DataFrame([geohash_decode(g) for g in all_gh], index=all_gh, columns=['lat','lon'])
train  = train.merge(coord, left_on='geohash', right_index=True, how='left')
test   = test .merge(coord, left_on='geohash', right_index=True, how='left')

# --- TRAIN-ONLY counts ---
gh_train_count   = train.groupby('geohash').size().rename('gh_train_count')
pfx5_train_count = train.groupby('gh_prefix5').size().rename('pfx5_train_count')

for d in (train, test):
    d['gh_train_count']   = d['geohash'].map(gh_train_count).fillna(0).astype(np.int32)
    d['pfx5_train_count'] = d['gh_prefix5'].map(pfx5_train_count).fillna(0).astype(np.int32)

cold_test_geohashes = sorted(set(test['geohash']) - set(train['geohash']))
print('Cold-start test geohashes:', len(cold_test_geohashes))

Cold-start test geohashes: 10


## 6. Modal static features (train-only, cold-start safe)

In [9]:
MODAL_COLS = ['RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks']

gh_modes = {col: train.groupby('geohash')[col].agg(det_mode) for col in MODAL_COLS}
pfx5_modes = {col: train.groupby('gh_prefix5')[col].agg(det_mode) for col in MODAL_COLS}
pfx4_modes = {col: train.groupby('gh_prefix4')[col].agg(det_mode) for col in MODAL_COLS}
global_modes = {col: det_mode(train[col]) for col in MODAL_COLS}
print('global modes:', global_modes)

modal_overrides = {gh: {} for gh in cold_test_geohashes}
for gh in cold_test_geohashes:
    for col in MODAL_COLS:
        v = pfx5_modes[col].get(gh[:5])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = pfx4_modes[col].get(gh[:4])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = global_modes[col]
        modal_overrides[gh][col] = v

def apply_modal(df):
    df = df.copy()
    for col in MODAL_COLS:
        modal_col = df['geohash'].map(gh_modes[col])
        for gh, mp in modal_overrides.items():
            modal_col = modal_col.where(df['geohash'] != gh, mp[col])
        df[f'{col}_modal'] = modal_col
    return df

train = apply_modal(train)
test  = apply_modal(test)

# NumberofLanes_modal -> integer
for d in (train, test):
    d['NumberofLanes_modal'] = d['NumberofLanes_modal'].fillna(d['NumberofLanes']).astype(np.int16)

for col in MODAL_COLS:
    assert train[f'{col}_modal'].notna().all()
    assert test[f'{col}_modal'].notna().all()
print('Modal columns populated for all rows.')

global modes: {'RoadType': 'Residential', 'NumberofLanes': '1', 'LargeVehicles': 'Not Allowed', 'Landmarks': 'Yes'}
Modal columns populated for all rows.


## 7. Categorical encoding (label → integer, vocab = train ∪ test)
We persist `vocab_sizes` so the transformer can size its embedding tables.

In [10]:
CAT_COLS = [
    'geohash', 'gh_prefix4', 'gh_prefix5',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'tod_bucket',
    'RoadType_modal', 'LargeVehicles_modal', 'Landmarks_modal',
]
encoders = {}
vocab_sizes = {}
for col in CAT_COLS:
    s_tr = train[col].astype(str)
    s_te = test [col].astype(str)
    cats = pd.Index(sorted(pd.concat([s_tr, s_te]).unique()))
    mapping = {v: i for i, v in enumerate(cats)}
    train[f'{col}_code'] = s_tr.map(mapping).astype(np.int32)
    test [f'{col}_code'] = s_te.map(mapping).astype(np.int32)
    encoders[col] = mapping
    vocab_sizes[f'{col}_code'] = len(cats)

# Plain 0/1 versions of binary categoricals (convenient for transformer)
for d in (train, test):
    d['LargeVehicles_bin']       = (d['LargeVehicles'] == 'Allowed').astype(np.int8)
    d['Landmarks_bin']           = (d['Landmarks'] == 'Yes').astype(np.int8)
    d['LargeVehicles_modal_bin'] = (d['LargeVehicles_modal'] == 'Allowed').astype(np.int8)
    d['Landmarks_modal_bin']     = (d['Landmarks_modal'] == 'Yes').astype(np.int8)

print('Vocab sizes:')
for k, v in vocab_sizes.items():
    print(f'  {k:<30} {v}')

Vocab sizes:
  geohash_code                   1259
  gh_prefix4_code                6
  gh_prefix5_code                56
  RoadType_code                  3
  LargeVehicles_code             2
  Landmarks_code                 2
  Weather_code                   4
  tod_bucket_code                5
  RoadType_modal_code            3
  LargeVehicles_modal_code       2
  Landmarks_modal_code           2


## 8. Build dense (geohash × t_abs) grid for causal lag/rolling features

Many geohashes have gaps in their 15-minute grid. We reindex onto the full `t_abs ∈ [4608, 4759]` (152 slots) per geohash so `shift(k)` gives true k-slots-ago, not k-observations-ago.

In [11]:
train['is_train'] = 1
test['is_train']  = 0
test['demand']        = np.nan
test['demand_log1p']  = np.nan
train['demand_log1p'] = np.log1p(train['demand'])

train = train.rename(columns={'Index': 'row_id'})
test  = test .rename(columns={'Index': 'row_id'})
df = pd.concat([train, test], ignore_index=True, sort=False)
print('Combined:', df.shape)

T_ABS_TRAIN_MAX = 4712
T_ABS_LO, T_ABS_HI = 4608, 4759
ALL_T_ABS = np.arange(T_ABS_LO, T_ABS_HI + 1)
geohashes = df['geohash'].unique()
print(f'Dense grid: t_abs {T_ABS_LO}..{T_ABS_HI} ({len(ALL_T_ABS)} slots), {len(geohashes)} geohashes')

gh_idx = pd.MultiIndex.from_product([geohashes, ALL_T_ABS], names=['geohash','t_abs'])
grid = pd.DataFrame(index=gh_idx).reset_index()
grid = grid.merge(df[['geohash','t_abs','demand_log1p','is_train']], on=['geohash','t_abs'], how='left')
grid['was_observed'] = grid['demand_log1p'].notna().astype(np.int8)
grid = grid.sort_values(['geohash','t_abs'], kind='stable').reset_index(drop=True)
print('Grid shape:', grid.shape, ' observed cells:', int(grid['was_observed'].sum()))

Combined: (118477, 50)
Dense grid: t_abs 4608..4759 (152 slots), 1259 geohashes
Grid shape: (191368, 5)  observed cells: 76699


## 9. Causal lag features

In [12]:
LAGS = [1, 2, 4, 8, 16, 96]
g = grid.groupby('geohash', sort=False)
for k in LAGS:
    grid[f'lag_{k}']         = g['demand_log1p'].shift(k)
    grid[f'was_obs_lag_{k}'] = g['was_observed'].shift(k).fillna(0).astype(np.int8)
print('Lag features computed.')

Lag features computed.


## 10. Rolling and EWM features (strictly causal)

In [13]:
for w in [4, 8, 16, 32]:
    grid[f'rolling_mean_{w}'] = g['demand_log1p'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean())
for w in [8, 32]:
    grid[f'rolling_std_{w}']  = g['demand_log1p'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=2).std())
grid['rolling_max_16'] = g['demand_log1p'].transform(
    lambda s: s.shift(1).rolling(16, min_periods=1).max())
grid['ewm_demand'] = g['demand_log1p'].transform(
    lambda s: s.shift(1).ewm(alpha=0.3, min_periods=1).mean())
print('Rolling/EWM features computed.')

Rolling/EWM features computed.


## 11. Boundary features (per-geohash, broadcast)

In [14]:
grid_train_portion = grid[grid['t_abs'] <= T_ABS_TRAIN_MAX]
records = []
for gh, sub in grid_train_portion.groupby('geohash', sort=False):
    obs = sub[sub['was_observed'] == 1].sort_values('t_abs')
    if len(obs) == 0:
        rec = dict(geohash=gh, boundary_demand=np.nan,
                   boundary_recency_1h_mean=np.nan,
                   boundary_recency_2h_mean=np.nan,
                   boundary_recency_4h_mean=np.nan,
                   saturation_fraction_last_24h=0.0,
                   train_history_len=0, last_observed_t_abs=np.nan)
    else:
        last96_raw = np.expm1(obs['demand_log1p'].iloc[-96:].values)
        rec = dict(
            geohash=gh,
            boundary_demand              = obs['demand_log1p'].iloc[-1],
            boundary_recency_1h_mean     = obs['demand_log1p'].iloc[-4:].mean(),
            boundary_recency_2h_mean     = obs['demand_log1p'].iloc[-8:].mean(),
            boundary_recency_4h_mean     = obs['demand_log1p'].iloc[-16:].mean(),
            saturation_fraction_last_24h = (last96_raw >= 0.99).mean() if len(last96_raw) else 0.0,
            train_history_len            = len(obs),
            last_observed_t_abs          = obs['t_abs'].iloc[-1],
        )
    records.append(rec)
boundary = pd.DataFrame(records)
print(boundary['train_history_len'].describe())

count    1259.000000
mean       60.920572
std        35.698081
min         0.000000
25%        28.000000
50%        70.000000
75%        93.000000
max       105.000000
Name: train_history_len, dtype: float64


## 12. Merge grid + boundary features onto df

In [15]:
grid_feature_cols = (
    ['was_observed']
    + [f'lag_{k}' for k in LAGS]
    + [f'was_obs_lag_{k}' for k in LAGS]
    + [f'rolling_mean_{w}' for w in [4,8,16,32]]
    + [f'rolling_std_{w}'  for w in [8,32]]
    + ['rolling_max_16', 'ewm_demand']
)
df = df.merge(grid[['geohash','t_abs'] + grid_feature_cols], on=['geohash','t_abs'], how='left')
df = df.merge(boundary, on='geohash', how='left')

df['cold_start_flag']           = (df['train_history_len'] == 0).astype(np.int8)
df['last_seen_gap_to_boundary'] = (df['t_abs'] - df['last_observed_t_abs']).astype('float32')
df['time_since_boundary']       = (df['t_abs'] - T_ABS_TRAIN_MAX).clip(lower=0).astype(np.int16)
df = df.drop(columns=['last_observed_t_abs'])
print('df after lag+boundary merge:', df.shape)

df after lag+boundary merge: (118477, 80)


## 13. 5-fold time-block CV + OOF empirical-Bayes target encodings

In [16]:
N_FOLDS, BLOCK_SIZE = 5, 21
fold_ranges = [(k+1, T_ABS_LO + k*BLOCK_SIZE, T_ABS_LO + k*BLOCK_SIZE + BLOCK_SIZE - 1)
               for k in range(N_FOLDS)]
for fid, lo, hi in fold_ranges:
    print(f'Fold {fid}: t_abs [{lo}, {hi}]')
assert fold_ranges[-1][2] == T_ABS_TRAIN_MAX

train_df = df[df['is_train'] == 1].copy()
def assign_fold(t):
    for fid, lo, hi in fold_ranges:
        if lo <= t <= hi: return fid
    return -1
train_df['fold_id'] = train_df['t_abs'].apply(assign_fold).astype(np.int8)
assert (train_df['fold_id'] >= 1).all()
print('Rows per fold:')
print(train_df['fold_id'].value_counts().sort_index())

Fold 1: t_abs [4608, 4628]
Fold 2: t_abs [4629, 4649]
Fold 3: t_abs [4650, 4670]
Fold 4: t_abs [4671, 4691]
Fold 5: t_abs [4692, 4712]
Rows per fold:
fold_id
1    17841
2    18649
3    17033
4     8211
5    14965
Name: count, dtype: int64


In [17]:
GLOBAL_MEAN_DEMAND = train_df['demand'].mean()
print(f'Global mean demand: {GLOBAL_MEAN_DEMAND:.6f}')

ENCODINGS = [
    ('gh_mean_demand_oof',      ['geohash'],                          'mean',   10),
    ('gh_p90_demand_oof',       ['geohash'],                          'p90',    10),
    ('gh_max_demand_oof',       ['geohash'],                          'max',    10),
    ('gh_std_demand_oof',       ['geohash'],                          'std',    10),
    ('gh_median_demand_oof',    ['geohash'],                          'median', 10),
    ('pfx5_mean_demand_oof',    ['gh_prefix5'],                       'mean',    5),
    ('pfx5_p90_demand_oof',     ['gh_prefix5'],                       'p90',     5),
    ('pfx4_mean_demand_oof',    ['gh_prefix4'],                       'mean',    5),
    ('gh_qhour_mean_oof',       ['geohash','qhour_of_day'],           'mean',    8),
    ('pfx5_qhour_mean_oof',     ['gh_prefix5','qhour_of_day'],        'mean',    5),
    ('qhour_roadtype_mean_oof', ['qhour_of_day','RoadType_code'],     'mean',    5),
    ('qhour_lanes_mean_oof',    ['qhour_of_day','NumberofLanes_modal'],'mean',   5),
]
AGG_FUNCS = {
    'mean':   lambda s: s.mean(),
    'p90':    lambda s: s.quantile(0.9),
    'median': lambda s: s.median(),
    'max':    lambda s: s.max(),
    'std':    lambda s: s.std(ddof=0),
}
def eb_encode(enc_data, group_cols, agg, k, global_value):
    gg = enc_data.groupby(group_cols)['demand']
    stat = gg.agg(AGG_FUNCS[agg]).rename('stat')
    cnt  = gg.size().rename('cnt')
    out = pd.concat([stat, cnt], axis=1).reset_index()
    out['eb'] = (out['cnt'] * out['stat'].fillna(global_value) + k * global_value) / (out['cnt'] + k)
    return out[group_cols + ['eb']]

Global mean demand: 0.093909


In [18]:
train_oof = train_df[['row_id'] + sorted({c for spec in ENCODINGS for c in spec[1]})].copy()
train_oof = train_oof.merge(train_df[['row_id','fold_id']], on='row_id', how='left')

for name, group_cols, agg, k in ENCODINGS:
    train_oof[name] = np.nan
    for fid, _, _ in fold_ranges:
        enc_data = train_df[train_df['fold_id'] != fid]
        enc_table = eb_encode(enc_data, group_cols, agg, k, GLOBAL_MEAN_DEMAND)
        rows_to_fill = train_oof['fold_id'] == fid
        merged = train_oof.loc[rows_to_fill, group_cols].merge(enc_table, on=group_cols, how='left')
        train_oof.loc[rows_to_fill, name] = merged['eb'].fillna(GLOBAL_MEAN_DEMAND).values

oof_cols = [spec[0] for spec in ENCODINGS]
assert train_oof[oof_cols].isna().sum().sum() == 0
print('Train OOF encodings — all 0 NaN ✓')

test_df = df[df['is_train'] == 0].copy()
test_enc_cols = sorted({c for spec in ENCODINGS for c in spec[1]})
test_enc_df = test_df[['row_id'] + test_enc_cols].copy()
for name, group_cols, agg, k in ENCODINGS:
    enc_table = eb_encode(train_df, group_cols, agg, k, GLOBAL_MEAN_DEMAND)
    merged = test_enc_df[group_cols].merge(enc_table, on=group_cols, how='left')
    test_enc_df[name] = merged['eb'].fillna(GLOBAL_MEAN_DEMAND).values
print('Test full-train encodings ✓')

df = df.merge(train_oof[['row_id'] + oof_cols], on='row_id', how='left')
test_to_attach = test_enc_df[['row_id'] + oof_cols].set_index('row_id')
df = df.set_index('row_id')
for c in oof_cols:
    df[c] = df[c].where(df.index.isin(train_df['row_id']), test_to_attach[c].reindex(df.index))
df = df.reset_index()
for c in oof_cols:
    assert df.loc[df['is_train']==1, c].isna().sum() == 0
    assert df.loc[df['is_train']==0, c].isna().sum() == 0
print('All encoded columns have 0 NaN ✓')

Train OOF encodings — all 0 NaN ✓
Test full-train encodings ✓
All encoded columns have 0 NaN ✓


## 14. Saturation features

In [19]:
saturating_geohashes = sorted(
    train_df.loc[train_df['demand'] >= 0.99, 'geohash'].unique().tolist())
print(f'Saturating geohashes: {len(saturating_geohashes)}')

df['is_saturated_geohash'] = df['geohash'].isin(saturating_geohashes).astype(np.int8)

GLOBAL_SAT_RATE = float((train_df['demand'] >= 0.99).mean())
sat_per_gh = (train_df.assign(_sat=(train_df['demand']>=0.99).astype(float))
                       .groupby('geohash')['_sat'].agg(['mean','count']))
sat_per_gh['eb'] = (sat_per_gh['count']*sat_per_gh['mean'] + 10*GLOBAL_SAT_RATE) / (sat_per_gh['count'] + 10)
df['gh_saturation_rate_oof'] = df['geohash'].map(sat_per_gh['eb']).fillna(GLOBAL_SAT_RATE).astype('float32')
df['highway_multilane'] = ((df['RoadType_modal']=='Highway') & (df['NumberofLanes_modal']>=2)).astype(np.int8)
print('Saturation features added.')

Saturating geohashes: 23
Saturation features added.


## 15. Sample weight

In [20]:
df['sample_weight'] = np.where(
    df['tod_bucket'].isin(['afternoon_drop','evening_trough']), 0.4, 1.0).astype(np.float32)
print(df.groupby('tod_bucket')['sample_weight'].first())

tod_bucket
afternoon_drop     0.4
evening_trough     0.4
late_night_rise    1.0
midday_plateau     1.0
morning_ramp       1.0
Name: sample_weight, dtype: float32


## 16. Numerical normalization (StandardScaler, fitted on TRAIN only)

Lag/rolling/boundary features can be NaN for cold-start or early-history geohashes. Before scaling we fill NaN with the **train-mean** of each column (transformer-friendly), and we add an `_isnan` companion flag for each filled feature so the model can learn to discount imputed values.

In [21]:
CONTINUOUS_COLS = (
    ['Temperature', 'lat', 'lon',
     'gh_train_count', 'pfx5_train_count',
     'NumberofLanes', 'NumberofLanes_modal',
     'last_seen_gap_to_boundary', 'time_since_boundary', 'train_history_len',
     'saturation_fraction_last_24h',
     'boundary_demand', 'boundary_recency_1h_mean',
     'boundary_recency_2h_mean', 'boundary_recency_4h_mean',
     'rolling_max_16', 'ewm_demand', 'gh_saturation_rate_oof']
    + [f'lag_{k}' for k in LAGS]
    + [f'rolling_mean_{w}' for w in [4,8,16,32]]
    + [f'rolling_std_{w}'  for w in [8,32]]
    + oof_cols
)

# Add NaN indicators + fill NaN with train mean BEFORE scaling
nan_companion_cols = []
train_mask = df['is_train'] == 1
for c in CONTINUOUS_COLS:
    if df[c].isna().any():
        df[f'{c}_isnan'] = df[c].isna().astype(np.int8)
        nan_companion_cols.append(f'{c}_isnan')
        fill_val = df.loc[train_mask, c].mean()
        df[c] = df[c].fillna(fill_val)
print(f'Added {len(nan_companion_cols)} *_isnan companion flags.')

scaler = StandardScaler()
scaler.fit(df.loc[train_mask, CONTINUOUS_COLS].values)
df[CONTINUOUS_COLS] = scaler.transform(df[CONTINUOUS_COLS].values).astype(np.float32)

# Sanity: train portion has mean≈0, std≈1
tr_check = df.loc[train_mask, CONTINUOUS_COLS]
print(f'Post-scale train means abs max: {tr_check.mean().abs().max():.4f} (expect ~0)')
print(f'Post-scale train stds  range  : [{tr_check.std().min():.3f}, {tr_check.std().max():.3f}] (expect ~1)')

Added 19 *_isnan companion flags.
Post-scale train means abs max: 0.0000 (expect ~0)
Post-scale train stds  range  : [0.000, 1.000] (expect ~1)


## 17. Final assemble and split

In [22]:
BINARY_COLS = [
    'LargeVehicles_bin', 'Landmarks_bin',
    'LargeVehicles_modal_bin', 'Landmarks_modal_bin',
    'RoadType_was_missing', 'Temperature_was_missing', 'Weather_was_missing',
    'cold_start_flag', 'is_saturated_geohash', 'highway_multilane',
    'was_observed',
] + [f'was_obs_lag_{k}' for k in LAGS] + nan_companion_cols

CYCLIC_COLS = ['hour_sin','hour_cos','qhour_sin','qhour_cos']
EMBEDDING_COLS = [f'{c}_code' for c in CAT_COLS]

KEEP_COLS = (
    ['row_id', 'geohash', 't_abs']
    + EMBEDDING_COLS + CYCLIC_COLS + CONTINUOUS_COLS + BINARY_COLS
    + ['sample_weight']
)
TRAIN_KEEP = KEEP_COLS + ['demand', 'demand_log1p']

train_features = (df[df['is_train']==1][TRAIN_KEEP]
                    .sort_values('row_id').reset_index(drop=True)
                    .rename(columns={'row_id':'Index'}))
test_features  = (df[df['is_train']==0][KEEP_COLS]
                    .sort_values('row_id').reset_index(drop=True)
                    .rename(columns={'row_id':'Index'}))
print('train_features:', train_features.shape)
print('test_features :', test_features.shape)

assert len(test_features) == 41778
assert train_features[CONTINUOUS_COLS + BINARY_COLS + EMBEDDING_COLS].isna().sum().sum() == 0
assert test_features [CONTINUOUS_COLS + BINARY_COLS + EMBEDDING_COLS].isna().sum().sum() == 0
print('All asserts pass ✓')

train_features: (76699, 99)
test_features : (41778, 97)
All asserts pass ✓


## 18. Save artifacts

In [ ]:
# Quick fix - convert period columns to float before saving
train_features['demand_log1p'] = train_features['demand_log1p'].astype(float)
test_features['demand_log1p'] = test_features['demand_log1p'].astype(float) if 'demand_log1p' in test_features.columns else test_features

train_features.to_parquet(os.path.join(PROCESSED, 'jyo_train_features.parquet'), index=False)
test_features.to_parquet(os.path.join(PROCESSED, 'jyo_test_features.parquet'), index=False)

# train_features.to_parquet(os.path.join(PROCESSED, 'jyo_train_features.parquet'), index=False)
# test_features.to_parquet (os.path.join(PROCESSED, 'jyo_test_features.parquet'),  index=False)

with open(os.path.join(PROCESSED, 'jyo_scalers.pkl'), 'wb') as f:
    pickle.dump({'standard_scaler': scaler,
                 'continuous_cols': CONTINUOUS_COLS,
                 'encoders': encoders}, f)

meta = {
    'embedding_cols':              EMBEDDING_COLS,
    'vocab_sizes':                 vocab_sizes,
    'continuous_cols':             CONTINUOUS_COLS,
    'cyclic_cols':                 CYCLIC_COLS,
    'binary_cols':                 BINARY_COLS,
    'nan_companion_cols':          nan_companion_cols,
    'sample_weight_col':           'sample_weight',
    'id_col':                      'Index',
    'target_col':                  'demand',
    'target_log_col':              'demand_log1p',
    'train_rows_dropped_roadtype': int(n_dropped),
    'train_t_abs_range':           [T_ABS_LO, T_ABS_TRAIN_MAX],
    'test_t_abs_range':            [T_ABS_TRAIN_MAX+1, T_ABS_HI],
    'fold_definition':             [{'fold_id': fid, 't_abs_lo': lo, 't_abs_hi': hi}
                                    for fid, lo, hi in fold_ranges],
    'saturating_geohash_count':    len(saturating_geohashes),
    'cold_test_geohash_count':     len(cold_test_geohashes),
}
with open(os.path.join(PROCESSED, 'jyo_feature_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fn in ['jyo_train_features.parquet','jyo_test_features.parquet',
           'jyo_scalers.pkl','jyo_feature_meta.json']:
    p = os.path.join(PROCESSED, fn)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')

ValueError: Cannot set a DataFrame with multiple columns to the single column demand_log1p

In [30]:
# === COMPLETE FIX FOR PERIOD DTYPE ERROR ===

def convert_period_to_float(df):
    """Convert all period dtype columns to float"""
    df = df.copy()
    for col in df.columns:
        # Check for period dtype (has 'freq' attribute or contains 'period' in string representation)
        if hasattr(df[col].dtype, 'freq') or 'period' in str(df[col].dtype).lower():
            print(f"Converting period column: {col} (dtype: {df[col].dtype})")
            # Convert to numeric (if possible) or to float
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
    return df

# Also ensure no object dtypes that might cause issues
def clean_for_parquet(df):
    df = df.copy()
    for col in df.columns:
        # Convert period to float
        if hasattr(df[col].dtype, 'freq') or 'period' in str(df[col].dtype).lower():
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
        # Convert categorical to codes or string
        elif pd.api.types.is_categorical_dtype(df[col]):
            df[col] = df[col].astype(str)
        # Convert object to string
        elif df[col].dtype == 'object':
            df[col] = df[col].astype(str)
        # Ensure ints are proper ints (not period)
        elif 'int' in str(df[col].dtype) and 'period' not in str(df[col].dtype).lower():
            df[col] = df[col].astype('int64')
    return df

# Apply conversion
print("Converting train_features...")
train_features_clean = clean_for_parquet(train_features)
print("Converting test_features...")
test_features_clean = clean_for_parquet(test_features)

# Double-check no period columns remain
print("\nVerifying no period columns left:")
for col in train_features_clean.columns:
    if hasattr(train_features_clean[col].dtype, 'freq') or 'period' in str(train_features_clean[col].dtype).lower():
        print(f"  WARNING: Still found period column: {col} - {train_features_clean[col].dtype}")
        # Force conversion to float
        train_features_clean[col] = pd.to_numeric(train_features_clean[col], errors='coerce').astype(float)
        test_features_clean[col] = pd.to_numeric(test_features_clean[col], errors='coerce').astype(float)

# Save to Parquet with pyarrow engine explicitly
print("\nSaving to Parquet...")
train_features_clean.to_parquet(
    os.path.join(PROCESSED, 'jyo_train_features.parquet'), 
    index=False,
    engine='pyarrow'
)
test_features_clean.to_parquet(
    os.path.join(PROCESSED, 'jyo_test_features.parquet'), 
    index=False,
    engine='pyarrow'
)

# Save scalers and encoders
with open(os.path.join(PROCESSED, 'jyo_scalers.pkl'), 'wb') as f:
    pickle.dump({
        'standard_scaler': scaler,
        'continuous_cols': CONTINUOUS_COLS,
        'encoders': encoders
    }, f)

# Save metadata
meta = {
    'embedding_cols':              EMBEDDING_COLS,
    'vocab_sizes':                 vocab_sizes,
    'continuous_cols':             CONTINUOUS_COLS,
    'cyclic_cols':                 CYCLIC_COLS,
    'binary_cols':                 BINARY_COLS,
    'nan_companion_cols':          nan_companion_cols,
    'sample_weight_col':           'sample_weight',
    'id_col':                      'Index',
    'target_col':                  'demand',
    'target_log_col':              'demand_log1p',
    'train_rows_dropped_roadtype': int(n_dropped),
    'train_t_abs_range':           [T_ABS_LO, T_ABS_TRAIN_MAX],
    'test_t_abs_range':            [T_ABS_TRAIN_MAX+1, T_ABS_HI],
    'fold_definition':             [{'fold_id': fid, 't_abs_lo': lo, 't_abs_hi': hi}
                                    for fid, lo, hi in fold_ranges],
    'saturating_geohash_count':    len(saturating_geohashes),
    'cold_test_geohash_count':     len(cold_test_geohashes),
}
with open(os.path.join(PROCESSED, 'jyo_feature_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('\nSaved:')
for fn in ['jyo_train_features.parquet','jyo_test_features.parquet',
           'jyo_scalers.pkl','jyo_feature_meta.json']:
    p = os.path.join(PROCESSED, fn)
    if os.path.exists(p):
        print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')
    else:
        print(f'  {p}  [FAILED TO SAVE]')

Converting train_features...
Converting test_features...

Verifying no period columns left:

Saving to Parquet...


ArrowKeyError: A type extension with name pandas.period already defined

In [31]:
# Replace cell 18 with this CSV version:

# Save to CSV instead of Parquet
train_features.to_csv(os.path.join(PROCESSED, 'jyo_train_features.csv'), index=False)
test_features.to_csv(os.path.join(PROCESSED, 'jyo_test_features.csv'), index=False)

# Save scalers and encoders
with open(os.path.join(PROCESSED, 'jyo_scalers.pkl'), 'wb') as f:
    pickle.dump({
        'standard_scaler': scaler,
        'continuous_cols': CONTINUOUS_COLS,
        'encoders': encoders
    }, f)

# Save metadata
meta = {
    'embedding_cols':              EMBEDDING_COLS,
    'vocab_sizes':                 vocab_sizes,
    'continuous_cols':             CONTINUOUS_COLS,
    'cyclic_cols':                 CYCLIC_COLS,
    'binary_cols':                 BINARY_COLS,
    'nan_companion_cols':          nan_companion_cols,
    'sample_weight_col':           'sample_weight',
    'id_col':                      'Index',
    'target_col':                  'demand',
    'target_log_col':              'demand_log1p',
    'train_rows_dropped_roadtype': int(n_dropped),
    'train_t_abs_range':           [T_ABS_LO, T_ABS_TRAIN_MAX],
    'test_t_abs_range':            [T_ABS_TRAIN_MAX+1, T_ABS_HI],
    'fold_definition':             [{'fold_id': fid, 't_abs_lo': lo, 't_abs_hi': hi}
                                    for fid, lo, hi in fold_ranges],
    'saturating_geohash_count':    len(saturating_geohashes),
    'cold_test_geohash_count':     len(cold_test_geohashes),
}
with open(os.path.join(PROCESSED, 'jyo_feature_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for fn in ['jyo_train_features.csv', 'jyo_test_features.csv',
           'jyo_scalers.pkl', 'jyo_feature_meta.json']:
    p = os.path.join(PROCESSED, fn)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')

Saved:
  /home/jyo/Desktop/Projects/yeeyyeeniggaass/processed/jyo_train_features.csv  (48549.4 KB)
  /home/jyo/Desktop/Projects/yeeyyeeniggaass/processed/jyo_test_features.csv  (25013.8 KB)
  /home/jyo/Desktop/Projects/yeeyyeeniggaass/processed/jyo_scalers.pkl  (17.8 KB)
  /home/jyo/Desktop/Projects/yeeyyeeniggaass/processed/jyo_feature_meta.json  (3.9 KB)


## 19. Round-trip sanity check

In [ ]:
tr = pd.read_parquet(os.path.join(PROCESSED, 'jyo_train_features.parquet'))
te = pd.read_parquet(os.path.join(PROCESSED, 'jyo_test_features.parquet'))
raw_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

assert te.shape[0] == 41778
assert (te['Index'].values == raw_test['Index'].values).all(), 'test Index alignment broken'
print(f'train rows: {len(tr)}  (raw 77299 - {n_dropped} dropped = {77299-n_dropped})')
print(f'test rows : {len(te)}')
print()
print('Embedding cols and vocab sizes:')
for c in EMBEDDING_COLS:
    print(f'  {c:<28} vocab_size={vocab_sizes[c]}')
print()
print(f'Continuous features (scaled): {len(CONTINUOUS_COLS)}')
print(f'Binary features             : {len(BINARY_COLS)}')
print(f'Cyclic features             : {len(CYCLIC_COLS)}')
print('Round-trip OK ✓')

In [32]:
# Update cell 19 to read CSV instead of Parquet
tr = pd.read_csv(os.path.join(PROCESSED, 'jyo_train_features.csv'))
te = pd.read_csv(os.path.join(PROCESSED, 'jyo_test_features.csv'))
raw_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

assert te.shape[0] == 41778
assert (te['Index'].values == raw_test['Index'].values).all(), 'test Index alignment broken'
print(f'train rows: {len(tr)}  (raw 77299 - {n_dropped} dropped = {77299-n_dropped})')
print(f'test rows : {len(te)}')
print()
print('Embedding cols and vocab sizes:')
for c in EMBEDDING_COLS:
    if c in tr.columns:
        print(f'  {c:<28} vocab_size={vocab_sizes[c]}')
print()
print(f'Continuous features (scaled): {len(CONTINUOUS_COLS)}')
print(f'Binary features             : {len(BINARY_COLS)}')
print(f'Cyclic features             : {len(CYCLIC_COLS)}')
print('Round-trip OK ✓')

train rows: 76699  (raw 77299 - 600 dropped = 76699)
test rows : 41778

Embedding cols and vocab sizes:
  geohash_code                 vocab_size=1259
  gh_prefix4_code              vocab_size=6
  gh_prefix5_code              vocab_size=56
  RoadType_code                vocab_size=3
  LargeVehicles_code           vocab_size=2
  Landmarks_code               vocab_size=2
  Weather_code                 vocab_size=4
  tod_bucket_code              vocab_size=5
  RoadType_modal_code          vocab_size=3
  LargeVehicles_modal_code     vocab_size=2
  Landmarks_modal_code         vocab_size=2

Continuous features (scaled): 42
Binary features             : 36
Cyclic features             : 4
Round-trip OK ✓


## Summary

The transformer model should consume:
- **Embedding inputs**: `*_code` columns + `vocab_sizes` from `jyo_feature_meta.json`
- **Continuous inputs**: `CONTINUOUS_COLS` (already z-scored, NaN-filled, with `_isnan` companion flags)
- **Cyclic inputs**: `hour_sin/cos`, `qhour_sin/cos`
- **Binary inputs**: `BINARY_COLS`
- **Target**: `demand` (raw, evaluation scale) or `demand_log1p` (training scale)
- **Sample weight**: `sample_weight` (down-weights out-of-test-distribution buckets)

Inference reproducibility is guaranteed by `jyo_scalers.pkl` (StandardScaler + categorical encoder maps).